In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded successfully!")

/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Processor loaded successfully!


/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()
Loading checkpoint shards: 100%|██████████| 3/3 [03:24<00:00, 68.28s/it]

Model loaded successfully!


In [2]:
# ============ AMBER ============
AMBER_IMG_PATH = "../data/amber/image"
QUERY_GENERATIVE_PATH = "../evaluation/AMBER/data/query/query_generative.json"
OUTPUT_PATH = "../results/infer/amber/llava15/dola_low.json"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 256

# ============ DoLa CONFIG ============
DOLA_LAYERS = "low"
REPETITION_PENALTY = 1.2

In [3]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm

processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id = processor.tokenizer.eos_token_id


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def clean_output(text):
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text

In [4]:
def resolve_image_path(image_name):
    image_path = Path(AMBER_IMG_PATH) / image_name

    if image_path.exists():
        return image_path

    matches = list(Path(AMBER_IMG_PATH).rglob(image_name))

    if len(matches) == 0:
        raise FileNotFoundError(f"Cannot find {image_name} under {AMBER_IMG_PATH}")

    return matches[0]

def prepare_batch(batch):
    images = []
    prompts = []
    ids = []

    for item in batch:
        image_id = int(item["id"])
        image_name = item["image"]
        prompt = item["query"]

        image_path = resolve_image_path(image_name)
        image = Image.open(image_path).convert("RGB")

        hf_prompt = f"USER: <image>\n{prompt} ASSISTANT:"

        images.append(image)
        prompts.append(hf_prompt)
        ids.append(image_id)

    return ids, images, prompts

In [5]:
def generate_dola_batch(ids, images, prompts):
    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    # sanity check: this should be BATCH_SIZE except the final batch
    print_once = False
    if not hasattr(generate_dola_batch, "printed"):
        print_once = True
        generate_dola_batch.printed = True

    if print_once:
        print("input_ids batch shape:", inputs["input_ids"].shape)
        print("num images:", len(images))
        print("num prompts:", len(prompts))

    input_device = next(model.parameters()).device

    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,

            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,

            # DoLa decoding
            dola_layers=DOLA_LAYERS,
            repetition_penalty=REPETITION_PENALTY,
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    batch_results = []

    for image_id, output in zip(ids, outputs):
        batch_results.append({
            "id": image_id,
            "response": clean_output(output)
        })

    return batch_results

In [6]:
with open(QUERY_GENERATIVE_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

queries = [q for q in queries if 1 <= int(q["id"]) <= 1004]

print(f"Loaded {len(queries)} AMBER generative samples.")
print("Example:", queries[0])

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

results = []

for batch in tqdm(batch_list(queries, BATCH_SIZE), total=(len(queries) + BATCH_SIZE - 1) // BATCH_SIZE):
    ids, images, prompts = prepare_batch(batch)

    # one generate call per batch
    batch_results = generate_dola_batch(ids, images, prompts)

    results.extend(batch_results)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} responses to {OUTPUT_PATH}")

Loaded 1004 AMBER generative samples.
Example: {'id': 1, 'image': 'AMBER_1.jpg', 'query': 'Describe this image.'}


  0%|          | 0/63 [00:00<?, ?it/s]

input_ids batch shape: torch.Size([16, 593])
num images: 16
num prompts: 16


100%|██████████| 63/63 [1:18:31<00:00, 74.79s/it] 


Saved 1004 responses to ../results/infer/amber/llava15/dola_low.json
